# QMC.jl for Lebesgue Integration

Translated from the original QMC notebook `lebesgue_integration.ipynb`.

Shows how to use QMC.jl for integration problems that don't naturally
start as expectations — i.e. Lebesgue integrals ∫_[a,b] f(x) dx.

In [1]:
using QMC
using Statistics
using Printf

Problem 1:  ∫₀² x² dx = 8/3

Rewrite as: 2 ∫₀² (x²/2) dx  where we integrate x²/2 w.r.t. Uniform(0,2)
and multiply by the measure volume (2).

In [2]:
println("="^60)
println("Problem 1: ∫₀² x² dx = 8/3 ≈ 2.6667")
println("="^60)

abs_tol = 0.5
true_value = 8.0 / 3.0

Problem 1: ∫₀² x² dx = 8/3 ≈ 2.6667


2.6666666666666665

Using Lebesgue measure (volume = 2)

In [3]:
dd = IIDStdUniform(1; seed=7)
tm = Lebesgue(dd; lower_bound=0.0, upper_bound=2.0)
f = CustomFun(tm, x -> x[:, 1] .^ 2)
sc = CubMCCLT(f; abs_tol=abs_tol)
result = integrate(sc)
@printf("  IID MC:       %.4f  (error = %.2e, n = %d)\n",
        result.solution, abs(result.solution - true_value), result.data[:n])

dd = Lattice(1; randomize=true, seed=7)
tm = Lebesgue(dd; lower_bound=0.0, upper_bound=2.0)
f = CustomFun(tm, x -> x[:, 1] .^ 2)
sc = CubQMCLatticeG(f; abs_tol=abs_tol, n_init=2^6, n_reps=16)
result = integrate(sc)
@printf("  Lattice QMC:  %.4f  (error = %.2e, n/rep = %d)\n",
        result.solution, abs(result.solution - true_value), result.data[:n])
println()

  IID MC:       1.3354  (error = 1.33e+00, n = 2048)
  Lattice QMC:  1.3359  (error = 1.33e+00, n/rep = 64)



Problem 2:  ∫₀¹ ∫₀¹ ∫₀¹ exp(x₁ x₂ x₃) dx₁ dx₂ dx₃

True value ≈ 1.14649 (computed symbolically)

In [4]:
println("="^60)
println("Problem 2: ∫[0,1]³ exp(x₁ x₂ x₃) dx = 1.14649...")
println("="^60)

true_value_2 = 1.14649  # approximate

dd = IIDStdUniform(3; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, x -> exp.(x[:, 1] .* x[:, 2] .* x[:, 3]))
sc = CubMCCLT(f; abs_tol=0.005)
result = integrate(sc)
@printf("  IID MC:       %.5f  (error ≈ %.2e, n = %d)\n",
        result.solution, abs(result.solution - true_value_2), result.data[:n])

dd = DigitalNetB2(3; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, x -> exp.(x[:, 1] .* x[:, 2] .* x[:, 3]))
sc = CubQMCNetG(f; abs_tol=0.001, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Sobol' QMC:   %.5f  (error ≈ %.2e, n/rep = %d)\n",
        result.solution, abs(result.solution - true_value_2), result.data[:n])
println()

Problem 2: ∫[0,1]³ exp(x₁ x₂ x₃) dx = 1.14649...
  IID MC:       1.14765  (error ≈ 1.16e-03, n = 15360)
  Sobol' QMC:   1.14651  (error ≈ 1.94e-05, n/rep = 1024)



Problem 3:  ∫₋₁³ ∫₋₁³ sin(x₁ + x₂) dx = ?

Using Lebesgue measure with non-standard bounds.
Volume = (3 - (-1))² = 16

In [5]:
println("="^60)
println("Problem 3: ∫[-1,3]² sin(x₁ + x₂) dx")
println("="^60)

Problem 3: ∫[-1,3]² sin(x₁ + x₂) dx


Exact: ∫₋₁³∫₋₁³ sin(x+y) dxdy = [-cos(x+y)]... = 4(cos(-2) - cos(2) - cos(2) + cos(6))
= 4(2cos(-2) - cos(2) - cos(6))... let's just compute numerically

In [6]:
true_value_3 = 4.0 * (cos(-2.0) - cos(2.0)) - 4.0 * (cos(2.0) - cos(6.0))

dd = Lattice(2; randomize=true, seed=7)
tm = Lebesgue(dd; lower_bound=-1.0, upper_bound=3.0)
f = CustomFun(tm, x -> sin.(x[:, 1] .+ x[:, 2]))
sc = CubQMCLatticeG(f; abs_tol=0.05, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Lattice QMC:  %.4f  (n/rep = %d)\n", result.solution, result.data[:n])
println()

println("="^60)
println("Lebesgue integration demos completed!")

  Lattice QMC:  0.1895  (n/rep = 256)

Lebesgue integration demos completed!
